In [48]:
!pip install langchain
!pip install groq
!pip install pymupdf
!pip install langchain_groq
!pip install dotenv
!pip install sentence-transformers
!pip install transformers torch
!pip install gradio


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached gradio-6.30.0-py3-none-any.whl.metadata (16 kB)
  Using cached brotli-1.2.0-cp312-cp312-win_amd64.whl.metadata (6.3 kB)
  Using cached fastapi-0.143.3-py3-none-any.whl.metadata (26 kB)
  Using cached gradio_client-2.7.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached hf_gradio-0.4.1-py3-none-any.whl.metadata (428 bytes)
  Using cached pandas-3.0.6-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached pillow-12.3.0-cp312-cp312-win_amd64.whl.metadata (9.3 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached python_multipart-0.0.32-py3-none-any.whl.metadata (2.1 kB)
  Using cached pytz-2026.5-py2.py3-none-any.whl.metadata (22 kB)
  Using cached safehttpx-0.1.7-py3-none-any.whl.metadata (4.2 kB)
  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9.7 kB)
  Using cached starlette-1.8.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached tomlkit-0.14.0-py3-none


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import langchain
from langchain_core.tools import tool

In [5]:
lease_contract = "C:\\Users\\urja tendolkar\\Desktop\\projects\\ClauseIQ\\training_docs\\lease_doc(contradict).pdf"
sell_contract = "C:\\Users\\urja tendolkar\\Desktop\\projects\\ClauseIQ\\training_docs\\sell_doc.pdf"

### Pdf content extraction

In [6]:
import re
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [7]:
import pymupdf as fitz

def extract_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)

    pages = []

    for page_num, page in enumerate(doc):

        # Extract page text with coordinate sorting
        words = page.get_text("words", sort=True)
        
        lines = []
        current_line = []
        current_y = None

        for word in words:
            x0, y0, x1, y1, text, block_no, line_no, word_no = word

            # New line if vertical position changes by more than 3pt
            if current_y is None or abs(y0 - current_y) <= 3:
                current_line.append(word)
            else:
                current_line.sort(key=lambda w: w[0])
                lines.append(" ".join(w[4] for w in current_line))
                current_line = [word]

            current_y = y0

        if current_line:
            current_line.sort(key=lambda w: w[0])
            lines.append(" ".join(w[4] for w in current_line))

        # Filter running headers and standalone page numbers per page
        filtered_lines = []
        for line in lines:
            stripped = line.strip()
            # Remove recurring government model draft header
            if re.search(r'Final model draft agreed to by MoUD and DoLR', stripped, re.IGNORECASE):
                continue
            # Remove standalone page numbers (e.g. 1, 2, 3...)
            if re.fullmatch(r'\d{1,2}', stripped):
                continue
            filtered_lines.append(line)

        pages.append({
            "page": page_num + 1,
            "text": "\n".join(filtered_lines)
        })

    doc.close()

    return pages

# Choose active contract: "lease_deed" or "sale_deed"
active_doc_type = "lease_deed"
active_contract_path = lease_contract if active_doc_type == "lease_deed" else sell_contract

output = extract_pdf_text(active_contract_path)
text = "\n".join(page["text"] for page in output)
print(f"Extracted {len(output)} pages from: {active_contract_path}")
print(f"Total extracted characters: {len(text)}")

Extracted text: Final model draft agreed to by MoUD and DoLR
Note: -This is a model draft and may be customized according to individual
requirement.
LEASE DEED
This Lease deed is made and executed at (Name of place) on this ……………….day 20th of
…………, August ………… 2026
BETWEEN
……………………………….,s/od/o………………………………………., Mr. Rajesh Mehta Mr. Mahesh Mehta
r/o…………………….…………………… Kandivali, Mumbai, Maharashtra (hereinafter called the Lessor) of the one part.
AND
……………………………….,s/o Ms. Ananya Sharma d/o………………………………………., Mr. Rakesh Sharma
r/o…………………………………………… Hinjewadi, Pune, Maharashtra (hereinafter called the Lessee) of the other part.
The expression Lessor & Lessee shall mean and include the parties itself, their respective legal
heirs, executors, successors, administrators, legal representatives and assigns/nominees of their
respective part.
Whereas Lessor is an absolute owner and in possession of the property no. 502, ……………., Sunshine Residency
situated at
……………………………………………………measuring…………………….. And

In [8]:
from schema import Contract

In [9]:
print(Contract.schema_json(indent=2))  # Print the schema for the Contract model

{
  "$defs": {
    "Clause": {
      "description": "Represents a clause in a legal document.",
      "properties": {
        "title": {
          "description": "The title of the clause",
          "title": "Title",
          "type": "string"
        },
        "text": {
          "description": "The text content of the clause",
          "title": "Text",
          "type": "string"
        },
        "section": {
          "anyOf": [
            {
              "type": "string"
            },
            {
              "type": "null"
            }
          ],
          "default": null,
          "description": "The section of the contract where the clause is located (if seperately specified)",
          "title": "Section"
        }
      },
      "required": [
        "title",
        "text"
      ],
      "title": "Clause",
      "type": "object"
    }
  },
  "description": "Represents a legal contract consisting of multiple clauses.",
  "properties": {
    "title": {
      "descri

C:\Users\urja tendolkar\AppData\Local\Temp\ipykernel_27388\50242175.py:1: PydanticDeprecatedSince20: The `schema_json` method is deprecated; use `model_json_schema` and json.dumps instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  print(Contract.schema_json(indent=2))  # Print the schema for the Contract model


In [10]:
import os
import httpx
from langchain_groq import ChatGroq

groq_key = os.getenv("GROQ_KEY") or os.getenv("GROQ_API_KEY")
if groq_key:
    os.environ["GROQ_API_KEY"] = groq_key

client = ChatGroq(
    model="openai/gpt-oss-20b", 
    api_key=groq_key,
    http_client=httpx.Client(verify=False, timeout=60.0)
)


In [11]:
# Covenants boundary markers for both contract types
covenants_start = {
    "lease_deed": r"NOW THIS LEASE DEED WITNESS(?:ETH)? AS UNDER\s*:-?",
    "sale_deed": r"9\.\s*Conditions of Sale\s*:",
}

covenants_end = {
    "lease_deed": r"SCHEDULE OF THE LEASED PROPERTY\s*:",
    "sale_deed": r"IN WITNESS WHEREOF",
}

def cleanup(text):
    """
    Cleans government template artifacts, disclaimer notes, blank placeholders,
    Unicode replacement characters, and normalizes whitespace.
    """
    # Remove recurring model draft header if any remaining
    text = re.sub(r'(?i)\bFinal model draft agreed to by MoUD and DoLR\b', ' ', text)

    # Remove standard model draft disclaimer note
    text = re.sub(
        r'(?is)Note\s*:\s*-?\s*This is a model draft and may be customized.*?(?:requirement[s]?|cases)\.?',
        ' ',
        text
    )

    # Replace ellipsis (\u2026), replacement char (\ufffd), dots, and underscores with space
    text = re.sub(r'[\u2026\ufffd]', ' ', text)
    text = re.sub(r'\.{2,}', ' ', text)
    text = re.sub(r'_{2,}', ' ', text)

    # Remove dangling isolated dots between spaces (e.g. "By . Rs" -> "By Rs")
    text = re.sub(r'\s+\.\s+', ' ', text)

    # Normalize whitespace and newlines
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def extract_clauses(contract_text, doc_type):
    """
    Generalized section extractor for lease deeds and sale deeds.
    Separates document into metadata (preamble, schedule, execution) and clauses (operative covenants).
    """
    start_pat = covenants_start.get(doc_type)
    end_pat = covenants_end.get(doc_type)
    if not start_pat:
        raise ValueError(f"Unsupported document type: {doc_type}")

    start_match = re.search(start_pat, contract_text, re.IGNORECASE)
    if not start_match:
        raise ValueError(f"Could not find covenants start marker for {doc_type}")

    preamble = contract_text[:start_match.start()].strip()
    remainder = contract_text[start_match.end():].strip()

    if end_pat:
        end_match = re.search(end_pat, remainder, re.IGNORECASE)
        if end_match:
            covenants_text = remainder[:end_match.start()].strip()
            closing_text = remainder[end_match.start():].strip()
        else:
            covenants_text = remainder
            closing_text = ""
    else:
        covenants_text = remainder
        closing_text = ""

    # Combine preamble and property schedule/execution block into metadata
    metadata = cleanup(preamble)
    if closing_text:
        metadata += "\n\n" + cleanup(closing_text)

    # Segment covenants into individual clauses based on document structure
    if doc_type == "sale_deed" or re.search(r'(?:\n|^)\s*\([ivx]+\)\s+', covenants_text, re.IGNORECASE):
        # Matches roman numeral sub-clauses, parenthesized numbers, or section headers
        pattern = r'(?:\n\s*|^)(?=(?:\([ivx]+\)|\(\d+\)|\d+\.\s+|The buyer confirms\s*:))'
        raw_chunks = re.split(pattern, covenants_text, flags=re.IGNORECASE)
        chunks = [c.strip() for c in raw_chunks if c.strip() and len(c.strip()) > 10]

        clause_list = []
        prefix = ""
        for p in chunks:
            if re.match(r'^The buyer confirms\s*:', p, re.IGNORECASE):
                prefix = "The buyer confirms: "
                rest = re.sub(r'^The buyer confirms\s*:\s*', '', p, flags=re.IGNORECASE).strip()
                if rest:
                    clause_list.append(prefix + rest)
            elif prefix and re.match(r'^\([ivx]+\)', p, re.IGNORECASE):
                clause_list.append(prefix + p)
            else:
                clause_list.append(p)
    else:
        # Numbered clauses pattern (e.g. 1., 2., 3. ...)
        raw_chunks = re.split(r'(?:\n\s*|^)(?=\d+\.\s+)', covenants_text)
        clause_list = [re.sub(r'^\d+\.\s+', '', c).strip() for c in raw_chunks if c.strip() and len(c.strip()) > 10]

    clauses_dict = {
        f"clause_{i}": cleanup(clause)
        for i, clause in enumerate(clause_list, start=1)
    }

    return metadata, clauses_dict

In [12]:
metadata, clauses = extract_clauses(text, active_doc_type)

print("=== METADATA PREVIEW ===")
print(metadata[:400] + ("..." if len(metadata) > 400 else ""))

print(f"\n=== EXTRACTED CLAUSES (Total: {len(clauses)}) ===")
for cid, ctext in clauses.items():
    print(f"{cid}: {ctext}")

METADATA:
Final model draft agreed to by MoUD and DoLR Note: -This is a model draft and may be customized according to individual requirement. LEASE DEED This Lease deed is made and executed at (Name of place) on this day 20th of , August 2026 BETWEEN ,s/od/o , Mr. Rajesh Mehta Mr. Mahesh Mehta r/o Kandivali, Mumbai, Maharashtra (hereinafter called the Lessor) of the one part. AND ,s/o Ms. Ananya Sharma d/o , Mr. Rakesh Sharma r/o Hinjewadi, Pune, Maharashtra (hereinafter called the Lessee) of the other part. The expression Lessor & Lessee shall mean and include the parties itself, their respective legal heirs, executors, successors, administrators, legal representatives and assigns/nominees of their respective part. Whereas Lessor is an absolute owner and in possession of the property no. 502, , Sunshine Residency situated at measuring Andheri West, Mumbai, Maharashtra 1200 sq.ft (hereinafter referred to as the SAID PROPERTY). Revenue District Mumbai Suburban Sub-Registrar Office Andh

In [13]:
contract = {
    "contract_type": active_doc_type,
    "metadata": metadata,
    "clauses": clauses
}

#### Maintaining short term context leveraging vector embeddings and FAISS indexing

In [14]:
!pip install faiss-cpu sentence-transformers


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import faiss
import numpy as np

In [16]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1701.09it/s]


In [17]:
clause_ids = list(clauses.keys())
clause_texts = list(clauses.values())

embeddings = model.encode(
    clause_texts,
    normalize_embeddings=True
)

In [18]:
embeddings = np.array(embeddings, dtype="float32")

#### Indexing

In [19]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
faiss.normalize_L2(embeddings)

print(index.ntotal)

19


### Natural Language Inference Models for contradictory clause identification

In [20]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_IDS = {
    "deberta_mnli": "microsoft/deberta-base-mnli",
    # "legal_roberta": "Shoriful025/legal-nli-roberta-base"
}

models = {}
tokenizers = {}

for name, model_id in MODEL_IDS.items():
    tokenizers[name] = AutoTokenizer.from_pretrained(model_id)
    models[name] = AutoModelForSequenceClassification.from_pretrained(model_id)
    models[name].eval()

Loading weights: 100%|██████████| 200/200 [00:00<00:00, 6173.68it/s]
[transformers] DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-base-mnli
Key    | Status     |  | 
-------+------------+--+-
config | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
import torch.nn.functional as F

def predict_nli(model_name, premise, hypothesis):
    tokenizer = tokenizers[model_name]
    model = models[model_name]

    inputs = tokenizer(
        premise,
        hypothesis,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    with torch.no_grad():
        logits = model(**inputs).logits
        probabilities = F.softmax(logits, dim=-1)[0]

    id2label = model.config.id2label

    scores = {
        id2label[i].lower(): float(probabilities[i])
        for i in range(len(probabilities))
    }

    predicted_label = max(scores, key=scores.get)

    return {
        "label": predicted_label,
        "scores": scores,
        "contradiction_score": scores.get("contradiction", 0.0)
    }

In [22]:
@tool
def classify_clause_risk(clause):
    """Classify a contract clause as LOW, MEDIUM, or HIGH risk."""

    prompt = """
    You are a legal document analysis assistant.

    Classify the following contract clause into one of three risk
    categories: LOW, MEDIUM, or HIGH.

    Criteria:

    LOW:
    The clause is standard, commonly accepted, and does not appear
    to impose significant obligations, restrictions, or unusual
    conditions.

    MEDIUM:
    The clause contains obligations, restrictions, ambiguity, or
    potentially burdensome conditions that may require attention
    or clarification.

    HIGH:
    The clause imposes significant obligations, restrictions,
    liabilities, penalties, or other conditions that could have
    serious implications for a party.

    Important:
    - Base the classification only on the provided clause.
    - Do not assume facts that are not present in the clause.
    - Do not declare a clause illegal or legally invalid.
    - Flag potential concerns that may warrant clarification or
      professional legal review.
    - Do not assess contradictions with other clauses here.
      Contradictions are handled separately.

    Respond in the following format:

    clause: <clause text>
    risk_category: <LOW, MEDIUM, or HIGH>
    justification: <brief explanation of the classification>
    """

    response = client.invoke(
        prompt + f"\n\nClause:\n{clause}"
    )

    return response.content

In [23]:
@tool
def summarize_clause(clause: str) -> str:
    """Explain a contract clause in simple language for a non-legal reader."""

    prompt = """
    You are a legal document analysis assistant.

    Your major function is to simplify complex legal language into
    clear, understandable explanations for non-legal readers.

    During your explanation, ensure that you:
    1. Use plain language and avoid unnecessary legal jargon.
    2. Provide context for legal terms that must be used.
    3. Do not provide legal advice or opinions.
    4. Do not assume facts that are not present in the clause.
    5. Keep the explanation concise and focused on the clause.
    6. Use examples or analogies only when they genuinely clarify
       a complex concept.
    7. Maintain the original meaning and intent of the clause.
    8. If the clause refers to another clause or document section,
       explain the reference in an understandable way.

    Contract clause:
    """

    response = client.invoke(
        prompt + f"\n\n{clause}"
    )

    return response.content

In [24]:
clause_to_id = {
    text: clause_id
    for clause_id, text in clauses.items()
}

In [32]:
clause_to_idx = {
    clause_id: idx
    for idx, clause_id in enumerate(clause_ids)
}

In [43]:
import json
from langchain_core.tools import tool


def get_related_clauses(clause: str) -> dict:
    """Retrieve candidate clauses and identify direct contractual relationships."""

    clause_id = clause_to_id.get(clause)

    if clause_id is None:
        return {
            "primary_clause_id": None,
            "related_clauses": [],
            "error": "Could not find the given clause in the current contract."
        }

    clause_idx = clause_to_idx[clause_id]
    query_vector = embeddings[clause_idx].reshape(1, -1)

    # Retrieve the top-k most similar other clauses
    k = min(10, max(0, len(clause_ids) - 1))
    if k == 0:
        return {
            "primary_clause_id": clause_id,
            "related_clauses": []
        }

    scores, indices = index.search(query_vector, min(len(clause_ids), k + 1))

    candidates = []

    for idx, score in zip(indices[0], scores[0]):
        if idx < 0:
            continue

        retrieved_id = clause_ids[idx]

        if retrieved_id == clause_id:
            continue

        candidates.append({
            "clause_id": retrieved_id,
            "similarity": round(float(score), 4),
            "text": clauses[retrieved_id]
        })

        if len(candidates) == k:
            break

    prompt = f"""
You are analyzing clauses within the SAME contract.

Primary clause:
{json.dumps({"clause_id": clause_id, "text": clause}, ensure_ascii=False)}

Candidate clauses:
{json.dumps(candidates, ensure_ascii=False)}

Identify only candidates having a DIRECT contractual relationship
with the primary clause.

A direct relationship exists when the wording supports that a clause:
- provides necessary context;
- implements or gives effect to the primary clause;
- modifies, qualifies, limits, or completes an obligation or right;
- is necessary to interpret or apply the primary clause.

Do not mark clauses related merely because they concern the same
property, contract, party, or general legal subject.

Use only the supplied clause texts. Do not invent relationships.
Be conservative. Return an empty list if no direct relationships exist.

Return ONLY valid JSON in this exact format:
{{
  "related_clauses": [
    {{
      "clause_id": "clause_1",
      "reason": "Brief explanation grounded in both clause texts",
      "relationship_type": "modifies / implements / qualifies / context / other"
    }}
  ]
}}

Only return clause IDs from the candidate list.
"""

    response = client.invoke(prompt)
    content = response.content

    # Handle list-based content responses
    if isinstance(content, list):
        content = "".join(
            block.get("text", "")
            for block in content
            if isinstance(block, dict) and block.get("type") == "text"
        )

    content = str(content).strip()

    # Remove Markdown code fences, if present
    content = re.sub(r"^```(?:json)?\s*", "", content)
    content = re.sub(r"\s*```$", "", content)

    # Extract the JSON object if the model added surrounding text
    try:
        parsed = json.loads(content)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", content, flags=re.DOTALL)

        if not match:
            print("RAW LLM RESPONSE:", repr(content))
            return {
                "primary_clause_id": clause_id,
                "related_clauses": [],
                "error": "The LLM did not return valid JSON.",
                "raw_response": content
            }

        try:
            parsed = json.loads(match.group(0))
        except json.JSONDecodeError:
            print("RAW LLM RESPONSE:", repr(content))
            return {
                "primary_clause_id": clause_id,
                "related_clauses": [],
                "error": "Could not parse the JSON in the LLM response.",
                "raw_response": content
            }

    if not isinstance(parsed, dict) or not isinstance(
        parsed.get("related_clauses"), list
    ):
        print("UNEXPECTED LLM JSON:", repr(content))
        return {
            "primary_clause_id": clause_id,
            "related_clauses": [],
            "error": "The LLM response did not match the expected JSON schema.",
            "raw_response": content
        }

    # Validate IDs against the actual retrieved candidates
    candidate_map = {
        candidate["clause_id"]: candidate
        for candidate in candidates
    }

    related = []

    for item in parsed.get("related_clauses", []):
        related_id = item.get("clause_id")
        candidate = candidate_map.get(related_id)

        if candidate is None:
            continue

        related.append({
            "clause_id": related_id,
            "text": candidate["text"],
            "similarity": candidate["similarity"],
            "reason": item.get("reason", ""),
            "relationship_type": item.get("relationship_type", "other")
        })

    return {
        "primary_clause_id": clause_id,
        "related_clauses": related
    }


@tool
def find_related_clauses(clause: str) -> str:
    """Find clauses directly related to the given clause."""

    result = get_related_clauses(clause)
    return json.dumps(result, indent=2, ensure_ascii=False)

In [44]:
@tool
def check_clause_contradictions(clause: str) -> str:
    """Identify directly related clauses and check them for contradictions
    using DeBERTa Natural Language Inference."""

    # Stage 1: Retrieve clauses and filter for direct relationships
    related_data = get_related_clauses(clause)

    if related_data.get("error"):
        return json.dumps(related_data, indent=2, ensure_ascii=False)

    related_clauses = related_data["related_clauses"]

    if not related_clauses:
        return json.dumps({
            "primary_clause_id": related_data["primary_clause_id"],
            "related_clauses_checked": 0,
            "contradictions_found": 0,
            "contradictions": [],
            "message": "No directly related clauses were identified."
        }, indent=2, ensure_ascii=False)

    # Stage 2: Load the already initialized NLI model and tokenizer
    nli_model = models["deberta_mnli"]
    nli_tokenizer = tokenizers["deberta_mnli"]
    nli_model.eval()

    id2label = nli_model.config.id2label

    # Handle either integer or string keys in id2label
    label_by_idx = {
        int(i): str(label).lower()
        for i, label in id2label.items()
    }

    contradiction_indices = [
        i for i, label in label_by_idx.items()
        if "contradiction" in label
    ]

    if len(contradiction_indices) != 1:
        return json.dumps({
            "error": "Could not uniquely identify the contradiction label.",
            "model_labels": label_by_idx
        }, indent=2, ensure_ascii=False)

    contradiction_idx = contradiction_indices[0]
    device = next(nli_model.parameters()).device

    contradictions = []

    # Stage 3: Run NLI on every directly related clause
    for candidate in related_clauses:
        candidate_text = candidate["text"]

        inputs = nli_tokenizer(
            clause,                 # Premise: primary clause
            candidate_text,         # Hypothesis: related clause
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            logits = nli_model(**inputs).logits
            probabilities = F.softmax(logits, dim=-1)[0]

        predicted_idx = int(probabilities.argmax().item())
        predicted_label = label_by_idx[predicted_idx]

        contradiction_score = float(
            probabilities[contradiction_idx].item()
        )

        # Keep only clauses whose highest-probability label is contradiction
        if "contradiction" in predicted_label:
            contradictions.append({
                "clause_id": candidate["clause_id"],
                "text": candidate_text,
                "relationship_reason": candidate["reason"],
                "relationship_type": candidate["relationship_type"],
                "similarity": candidate["similarity"],
                "nli_label": predicted_label,
                "contradiction_score": round(contradiction_score, 4)
            })

    # Stage 4: Sort by contradiction score, highest first
    contradictions.sort(
        key=lambda item: item["contradiction_score"],
        reverse=True
    )

    return json.dumps({
        "primary_clause_id": related_data["primary_clause_id"],
        "related_clauses_checked": len(related_clauses),
        "contradictions_found": len(contradictions),
        "contradictions": contradictions
    }, indent=2, ensure_ascii=False)

In [ ]:
def contradictory_clause(clause: str, top_k_exclude: int = 7) -> list:
    """
    Identify clauses in the same contract that may contradict the input clause.
    Uses existing IndexFlatIP FAISS index for candidate selection and existing NLI model for classification.

    Parameters:
    - clause: The input clause text to evaluate.
    - top_k_exclude: Number of top similar *other* clauses to exclude (default: 7).

    Returns:
    - List of dicts sorted by descending NLI contradiction score with fields:
      - input_clause_id
      - input_clause_text
      - contradictory_clause_id
      - contradictory_clause_text
      - contradiction_score
      - nli_label
      - direction
    """
    input_cid = clause_to_id.get(clause)
    if input_cid is not None and input_cid in clause_to_idx:
        q_vec = embeddings[clause_to_idx[input_cid]].reshape(1, -1)
    else:
        q_vec = model.encode([clause], normalize_embeddings=True).astype("float32")
        input_cid = "input_clause"

    # Candidate selection using FAISS (IndexFlatIP with normalized vectors = cosine similarity)
    scores, indices = index.search(q_vec, len(clause_ids))

    # Exclude the input clause itself
    other_candidates = []
    for score, idx_val in zip(scores[0], indices[0]):
        if idx_val < 0:
            continue
        cid = clause_ids[idx_val]
        if cid == input_cid:
            continue
        other_candidates.append({
            "clause_id": cid,
            "text": clauses[cid],
            "similarity": float(score)
        })

    # Exclude the top 6-7 most similar other clauses
    eligible = other_candidates[top_k_exclude:] if len(other_candidates) > top_k_exclude else []

    # Contradiction classification using NLI
    nli_model = models.get("deberta_mnli")
    nli_tok = tokenizers.get("deberta_mnli")
    if nli_model is None or nli_tok is None:
        raise RuntimeError("NLI model 'deberta_mnli' is not loaded.")

    id2label = {int(k): str(v).lower() for k, v in nli_model.config.id2label.items()}
    contra_indices = [k for k, v in id2label.items() if "contradiction" in v]
    if not contra_indices:
        raise ValueError("Could not find contradiction label in NLI model configuration.")
    contra_idx = contra_indices[0]
    device = next(nli_model.parameters()).device

    contradictions = []
    for cand in eligible:
        cand_id = cand["clause_id"]
        cand_text = cand["text"]

        # Evaluate Direction 1: input -> candidate (premise -> hypothesis)
        inputs1 = nli_tok(clause, cand_text, truncation=True, max_length=512, return_tensors="pt")
        inputs1 = {k: v.to(device) for k, v in inputs1.items()}
        with torch.no_grad():
            p1 = F.softmax(nli_model(**inputs1).logits, dim=-1)[0]
        lbl1 = id2label[int(p1.argmax().item())]
        sc1 = float(p1[contra_idx].item())

        # Evaluate Direction 2: candidate -> input (candidate premise -> input hypothesis)
        inputs2 = nli_tok(cand_text, clause, truncation=True, max_length=512, return_tensors="pt")
        inputs2 = {k: v.to(device) for k, v in inputs2.items()}
        with torch.no_grad():
            p2 = F.softmax(nli_model(**inputs2).logits, dim=-1)[0]
        lbl2 = id2label[int(p2.argmax().item())]
        sc2 = float(p2[contra_idx].item())

        # Select only candidate pairs whose predicted NLI label is contradiction
        if "contradiction" in lbl1:
            contradictions.append({
                "input_clause_id": input_cid,
                "input_clause_text": clause,
                "contradictory_clause_id": cand_id,
                "contradictory_clause_text": cand_text,
                "contradiction_score": round(sc1, 4),
                "nli_label": lbl1,
                "direction": f"{input_cid} -> {cand_id}"
            })
        elif "contradiction" in lbl2:
            contradictions.append({
                "input_clause_id": input_cid,
                "input_clause_text": clause,
                "contradictory_clause_id": cand_id,
                "contradictory_clause_text": cand_text,
                "contradiction_score": round(sc2, 4),
                "nli_label": lbl2,
                "direction": f"{cand_id} -> {input_cid}"
            })

    # Sort descending by contradiction_score
    contradictions.sort(key=lambda item: item["contradiction_score"], reverse=True)
    return contradictions

In [ ]:
# Test contradictory_clause on clause_17
print("Evaluating contradictory_clause on clause_17...")
contra_results = contradictory_clause(clauses["clause_17"], top_k_exclude=7)
print(f"Identified {len(contra_results)} potential contradiction(s):")
for r in contra_results[:3]:
    print(f"  - Conflicting Clause: {r['contradictory_clause_id']} | Score: {r['contradiction_score']} | Direction: {r['direction']}")
    print(f"    Text: {r['contradictory_clause_text'][:85]}...")

In [46]:
result = check_clause_contradictions.invoke({
    "clause": clauses["clause_2"]
})

print(result)

{
  "primary_clause_id": "clause_2",
  "related_clauses_checked": 3,
  "contradictions_found": 1,
  "contradictions": [
    {
      "clause_id": "clause_8",
      "text": "That the Lessor acknowledges receipt of an interest free advance amount of Rs 18000 (Rupees only) Eighteen thousand by cash/cheque/DD. No dated drawn on. which will be refundable upon expiry of the lease or sooner determination on account of violation of the convents of the lease subject to receipt of vacant physical possession by the lessor and set-off of the claims if any of the lessor on account of breaches on part of the lessee.",
      "relationship_reason": "Clause 8 specifies an interest‑free advance amount of Rs 18000 and outlines its refund conditions, thereby providing the operational details and conditions that apply to the advance/premium mentioned in the primary clause.",
      "relationship_type": "implements",
      "similarity": 0.3808,
      "nli_label": "contradiction",
      "contradiction_score": 

In [34]:
from langchain.agents import create_agent

tools = [
    classify_clause_risk,
    summarize_clause,
    find_related_clauses
]

agent = create_agent(
    model=client,
    tools=tools,
    system_prompt="""
    You are ClauseIQ, a contract analysis assistant.

    Analyze contractual clauses for one or more of the following tasks:
    - Risk classification (LOW, MEDIUM, HIGH)
    - Simplified explanations for non-legal readers
    - Identification of related clauses within the contract (only if actually found)
    
    Note: DO NOT give any random related clauses if none are found.
    Structure the output as:
    Risk classification: {risk category}
    Simplified explanation: {explanation}
    Related clauses: {list of related clauses, with their content (if any)}
    
    Use the available tools when they are useful.
    Do not use all the tools for every clause; only use the tools that are relevant to the task at hand.
    Do not provide definitive legal advice.
    """
)

In [35]:
clause_id = 6

In [36]:
clauses

{'clause_1': 'Period of Lease : From To 01/09/2026 31/07/2027',
 'clause_2': 'Amount of Advance/Premium : Rs (Rupees 25000 Twenty five thousand only), (if any)',
 'clause_3': 'Security Deposit, if any : Rs 150000 (Rupees One Lakh Fifty only), thousand (if any)',
 'clause_4': 'Details of Periodicity & Mode of Payment of rent. Monthly rent of 45000 to be done using cheque or through net zbanking, by 7th of every month.',
 'clause_5': 'Stamp duty paid : By Rs (Rupees 100 Hundred only), (if any)',
 'clause_6': 'That the lessor is the sole/ joint/ co-owner of the said property being leased out and in possession of the said property mentioned in the Schedule below.',
 'clause_7': 'That the lessee has approached the lessor for grant of a lease of the property described in the schedule below for a period mentioned above and the lessor has agreed to grant the said property on lease on the terms and conditions hereinafter appearing.',
 'clause_8': 'That the Lessor acknowledges receipt of an inte

In [37]:
clauses[f'clause_{clause_id}']

'That the lessor is the sole/ joint/ co-owner of the said property being leased out and in possession of the said property mentioned in the Schedule below.'

In [38]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": f"""
            Give me related information I need to know about the following clause in this legal document.

            Clause: {clauses[f"clause_{clause_id}"]}
            Contract type: {contract['contract_type']}
            """
        }
    ]
})

PRIMARY: clause_6
CANDIDATES:
clause_12 0.7719 That the lessee shall not sub-let/ sub-lease any portion of the property or part thereof to any person without the prior permission of the lessor, in writing.
clause_7 0.748 That the lessee has approached the lessor for grant of a lease of the property described in the schedule below for a period mentioned above and the lessor has agreed to grant the said property on lease on the terms and conditions hereinafter appearing.
clause_15 0.7314 That at the end of the lease period, the property shall be handed over to the lessor in good condition. The lessee shall have the right to refund the advance/premium paid subject to set-off of any claim of the lessor.
clause_8 0.6752 That the Lessor acknowledges receipt of an interest free advance amount of Rs 18000 (Rupees only) Eighteen thousand by cash/cheque/DD. No dated drawn on. which will be refundable upon expiry of the lease or sooner determination on account of violation of the convents of the 

In [39]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage, SystemMessage
print(clauses[f"clause_{clause_id}"])
for msg in result["messages"]:
    if isinstance(msg, AIMessage):
        print(msg.content)

That the lessor is the sole/ joint/ co-owner of the said property being leased out and in possession of the said property mentioned in the Schedule below.


**Risk classification:** LOW  

**Simplified explanation:**  
This clause simply states that the person who is leasing the property (the lessor) is the owner or shares ownership of the property. It confirms that the lessor has legal title and control over the property that is being rented out. There are no additional obligations or restrictions imposed by this statement; it merely records the ownership status.  

**Related clauses:**  
None.


## LangGraph Dynamic Routing Agent with Dual Memory (STM & LTM)

This architecture organizes ClauseIQ as a multi-node state graph with:
- **Dual Memory**:
  - **Short-Term Memory (STM)**: Session conversation checkpointing (`MemorySaver`) + in-memory FAISS vector index of the current active document.
  - **Long-Term Memory (LTM)**: Standard market benchmark templates and recurring legal risk patterns.
- **Routing Dependency Hierarchy**:
  - **Non-FAISS track**: Plain English explanation (operates strictly on single input clause without FAISS).
  - **FAISS track**: Intra-contract cross-clause interactions (`Risk` -> `Contradiction` -> `FAISS`).
- **Reflection / Completeness Loop**: Checks if any requested user intent was missed; if so, re-routes to execute the missing path and merges results into a single compiled JSON debug response with zero redundancy.

In [ ]:
# Long-Term Memory (LTM): Persistent Legal Benchmarks and Risk Templates
LTM_BENCHMARKS = [
    {
        "id": "ltm_permitted_use",
        "category": "permitted_use",
        "title": "Standard Permitted Use Covenant",
        "text": "The Lessee shall use the Leased Premises solely and exclusively for the permitted lawful purpose and shall not use or permit the premises to be used for any unlawful, hazardous, offensive, or commercial trade without prior written consent.",
        "standard_risk_flags": ["Absence of purpose specification", "Unrestricted commercial use without lessor consent"]
    },
    {
        "id": "ltm_maintenance_repair",
        "category": "maintenance",
        "title": "Standard Maintenance and Structural Repair Covenant",
        "text": "The Lessor shall be responsible for major structural repairs and external maintenance, while the Lessee shall maintain the interior premises, sanitary fittings, and electrical installations in tenantable condition.",
        "standard_risk_flags": ["Ambiguity in structural vs day-to-day repair liability", "Unfair allocation of statutory tax hikes"]
    },
    {
        "id": "ltm_termination_notice",
        "category": "termination",
        "title": "Standard Termination and Notice Period",
        "text": "Either party may terminate this agreement prior to expiry by serving a minimum of one to three calendar months advance written notice or by paying rent in lieu thereof.",
        "standard_risk_flags": ["Immediate unilateral termination without notice", "Lock-in period conflicts with termination rights"]
    },
    {
        "id": "ltm_subletting_assignment",
        "category": "subletting",
        "title": "Standard Subletting and Assignment Restriction",
        "text": "The Lessee shall not assign, sublet, underlet, or part with possession of the premises or any portion thereof without the prior written consent of the Lessor.",
        "standard_risk_flags": ["Absolute subletting rights without landlord oversight", "Uncontrolled third-party occupancy"]
    },
    {
        "id": "ltm_security_deposit",
        "category": "deposit",
        "title": "Standard Security Deposit and Refund Covenant",
        "text": "The interest-free security deposit shall be refunded to the Lessee simultaneously upon handing over peaceful vacant possession of the premises, subject to lawful deductions for outstanding dues or damage.",
        "standard_risk_flags": ["Vague refund timelines", "Non-refundable deposits disguised as security"]
    }
]

# Precompute LTM benchmark embeddings using the existing SentenceTransformer model
ltm_texts = [b["text"] for b in LTM_BENCHMARKS]
ltm_embeddings = model.encode(ltm_texts, normalize_embeddings=True).astype("float32")
ltm_index = faiss.IndexFlatIP(ltm_embeddings.shape[1])
ltm_index.add(ltm_embeddings)

def search_ltm_benchmarks(clause_text: str, top_k: int = 1) -> dict:
    """Retrieve the closest market-standard benchmark from Long-Term Memory."""
    q_vec = model.encode([clause_text], normalize_embeddings=True).astype("float32")
    scores, indices = ltm_index.search(q_vec, top_k)
    best_idx = indices[0][0]
    best_score = float(scores[0][0])
    match = LTM_BENCHMARKS[best_idx]
    return {
        "benchmark_id": match["id"],
        "title": match["title"],
        "similarity": round(best_score, 4),
        "standard_text": match["text"],
        "standard_risk_flags": match["standard_risk_flags"]
    }

print(f"Initialized Long-Term Memory (LTM) with {len(LTM_BENCHMARKS)} benchmark legal patterns.")

In [ ]:
from typing import TypedDict, Optional, List, Dict, Any
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

class ClauseIQState(TypedDict):
    contract_type: str
    input_clause_id: str
    input_clause_text: str
    user_query: str
    thread_id: str
    
    # Execution & Routing Control
    intents: List[str]                  # ['explain', 'contradiction', 'risk']
    visited_nodes: List[str]            # Prevents redundant execution
    
    # Analysis outputs
    plain_summary: Optional[str]
    faiss_candidates: Optional[List[Dict[str, Any]]]
    contradictions: Optional[List[Dict[str, Any]]]
    ltm_benchmark: Optional[Dict[str, Any]]
    risk_assessment: Optional[Dict[str, Any]]
    final_response: Optional[Dict[str, Any]]

In [ ]:
import json
import re

def router_node(state: ClauseIQState) -> Dict[str, Any]:
    """Analyzes the user query and registers required intents."""
    visited = list(state.get("visited_nodes", []))
    visited.append("router_node")
    
    query = state.get("user_query", "").lower()
    intents = list(state.get("intents", []))
    
    if not intents:
        if any(w in query for w in ["explain", "simplify", "summary", "summarize", "understand", "meaning", "plain"]):
            intents.append("explain")
        if any(w in query for w in ["contradict", "conflict", "inconsistent", "oppose", "violate"]):
            intents.append("contradiction")
        if any(w in query for w in ["risk", "danger", "hazard", "threat", "liability", "flag", "caution"]):
            intents.append("risk")
            
        # Default to comprehensive review if no specific intent keywords matched
        if not intents:
            intents = ["explain", "contradiction", "risk"]
            
    return {"intents": intents, "visited_nodes": visited}

def explain_node(state: ClauseIQState) -> Dict[str, Any]:
    """Non-FAISS track: Generates a plain-English explanation of the single input clause."""
    visited = list(state.get("visited_nodes", []))
    visited.append("explain_node")
    
    clause_text = state.get("input_clause_text", "")
    prompt = f"""
    You are a legal document analysis assistant.
    Explain this contract clause in simple language for a non-legal reader.
    Maintain original meaning and avoid unnecessary legal jargon.
    
    Clause:
    {clause_text}
    """
    try:
        resp = client.invoke(prompt)
        content = resp.content.strip()
    except Exception as e:
        content = f"Plain language explanation for {state.get('input_clause_id')}: This clause establishes operative covenants concerning: '{clause_text[:120]}...'"
        
    return {"plain_summary": content, "visited_nodes": visited}

def faiss_node(state: ClauseIQState) -> Dict[str, Any]:
    """Short-Term Memory: Retrieves candidate clauses from the current document via FAISS."""
    visited = list(state.get("visited_nodes", []))
    visited.append("faiss_node")
    
    clause_text = state.get("input_clause_text", "")
    input_cid = state.get("input_clause_id", "")
    
    if input_cid in clause_to_idx:
        q_vec = embeddings[clause_to_idx[input_cid]].reshape(1, -1)
    else:
        q_vec = model.encode([clause_text], normalize_embeddings=True).astype("float32")
        
    scores, indices = index.search(q_vec, len(clause_ids))
    candidates = []
    for sc, i in zip(scores[0], indices[0]):
        if i < 0:
            continue
        cid = clause_ids[i]
        if cid == input_cid:
            continue
        candidates.append({
            "clause_id": cid,
            "text": clauses[cid],
            "similarity": round(float(sc), 4)
        })
        
    return {"faiss_candidates": candidates, "visited_nodes": visited}

def contradiction_node(state: ClauseIQState) -> Dict[str, Any]:
    """Evaluates candidates using the NLI model (via contradictory_clause)."""
    visited = list(state.get("visited_nodes", []))
    visited.append("contradiction_node")
    
    clause_text = state.get("input_clause_text", "")
    found_contradictions = contradictory_clause(clause_text, top_k_exclude=7)
    
    return {"contradictions": found_contradictions, "visited_nodes": visited}

def ltm_node(state: ClauseIQState) -> Dict[str, Any]:
    """Long-Term Memory: Retrieves standard market benchmarks and identifies deviations."""
    visited = list(state.get("visited_nodes", []))
    visited.append("ltm_node")
    
    clause_text = state.get("input_clause_text", "")
    ltm_match = search_ltm_benchmarks(clause_text, top_k=1)
    
    return {"ltm_benchmark": ltm_match, "visited_nodes": visited}

def risk_node(state: ClauseIQState) -> Dict[str, Any]:
    """Context-aware risk classification combining clause text, NLI contradictions, and LTM benchmarks."""
    visited = list(state.get("visited_nodes", []))
    visited.append("risk_node")
    
    clause_text = state.get("input_clause_text", "")
    contradictions = state.get("contradictions", []) or []
    ltm_match = state.get("ltm_benchmark", {})
    
    prompt = f"""
    You are a contract risk evaluation specialist.
    Classify the contractual risk of this clause into: LOW, MEDIUM, or HIGH.
    
    Primary Clause: {clause_text}
    Identified Intra-Contract Contradictions ({len(contradictions)}):
    {json.dumps(contradictions[:3], indent=2, ensure_ascii=False)}
    Market Benchmark Comparison:
    {json.dumps(ltm_match, indent=2, ensure_ascii=False)}
    
    Evaluate potential legal liabilities, operational tensions, and compliance concerns.
    Format your response as valid JSON:
    {{
      "risk_level": "LOW / MEDIUM / HIGH",
      "justification": "Clear, grounded explanation of the risk level",
      "flagged_issues": ["Specific bulleted risk points"]
    }}
    """
    try:
        resp = client.invoke(prompt)
        text_resp = re.sub(r"^```(?:json)?\s*", "", resp.content.strip())
        text_resp = re.sub(r"\s*```$", "", text_resp)
        match = re.search(r"\{.*\}", text_resp, re.DOTALL)
        data = json.loads(match.group(0)) if match else {"risk_level": "HIGH" if contradictions else "LOW", "justification": text_resp, "flagged_issues": []}
    except Exception as e:
        level = "HIGH" if contradictions else ("MEDIUM" if ltm_match.get("similarity", 0) < 0.3 else "LOW")
        data = {
            "risk_level": level,
            "justification": f"Automated risk evaluation determined {level} risk. Detected {len(contradictions)} potential intra-contract contradiction(s) and evaluated against benchmark: '{ltm_match.get('title')}'.",
            "flagged_issues": [f"Contradiction flagged with {c.get('contradictory_clause_id')}" for c in contradictions]
        }
        
    return {"risk_assessment": data, "visited_nodes": visited}

def compiler_node(state: ClauseIQState) -> Dict[str, Any]:
    """Compiles, synthesizes, and formats the unified analysis into structured JSON for debugging."""
    visited = list(state.get("visited_nodes", []))
    visited.append("compiler_node")
    
    clause_id = state.get("input_clause_id")
    clause_text = state.get("input_clause_text")
    risk_data = state.get("risk_assessment") or {}
    contradictions = state.get("contradictions") or []
    plain_summary = state.get("plain_summary")
    ltm_data = state.get("ltm_benchmark")
    
    synth_prompt = f"""
    Synthesize a concise 2-sentence executive takeaway for this contract clause review:
    Clause ID: {clause_id}
    Risk: {risk_data.get('risk_level', 'Not evaluated')}
    Contradictions: {len(contradictions)} identified
    Summary: {plain_summary or 'N/A'}
    """
    try:
        resp = client.invoke(synth_prompt)
        exec_summary = resp.content.strip()
    except Exception as e:
        exec_summary = f"Clause {clause_id} exhibits {risk_data.get('risk_level', 'evaluated')} risk with {len(contradictions)} potential intra-contract contradiction(s) flagged for legal verification."
        
    response = {
        "status": "success",
        "contract_type": state.get("contract_type"),
        "target_clause": {
            "id": clause_id,
            "text": clause_text
        },
        "executive_summary": exec_summary,
        "analysis": {
            "plain_explanation": plain_summary,
            "contradictions": contradictions,
            "long_term_benchmark": ltm_data,
            "risk_assessment": risk_data
        },
        "debug_trace": {
            "execution_flow": visited,
            "intents_requested": state.get("intents"),
            "total_nodes_executed": len(visited),
            "redundancy_prevented": True
        }
    }
    return {"final_response": response, "visited_nodes": visited}

In [ ]:
# Conditional Routing Functions
def route_from_router(state: ClauseIQState) -> str:
    """Dispatches from router to FAISS path or Explain path based on query intents."""
    visited = state.get("visited_nodes", [])
    intents = state.get("intents", [])
    
    if any(i in ["contradiction", "risk"] for i in intents) and "faiss_node" not in visited:
        return "faiss_node"
    if "explain" in intents and "explain_node" not in visited:
        return "explain_node"
    return "compiler_node"

def route_after_contradiction(state: ClauseIQState) -> str:
    """After contradiction analysis, routes to LTM/Risk if risk analysis was requested."""
    intents = state.get("intents", [])
    visited = state.get("visited_nodes", [])
    
    if "risk" in intents and "risk_node" not in visited:
        return "ltm_node"
    return "compiler_node"

def completeness_check(state: ClauseIQState) -> str:
    """Reflection loop: Verifies whether any requested intent was missed; loops back if incomplete."""
    visited = state.get("visited_nodes", [])
    intents = state.get("intents", [])
    
    # If explain was requested but not visited, re-route
    if "explain" in intents and "explain_node" not in visited:
        return "router_node"
    # If contradiction or risk was requested but faiss path not visited, re-route
    if any(i in ["contradiction", "risk"] for i in intents) and "faiss_node" not in visited:
        return "router_node"
        
    return END

# Assemble StateGraph
builder = StateGraph(ClauseIQState)

builder.add_node("router_node", router_node)
builder.add_node("explain_node", explain_node)
builder.add_node("faiss_node", faiss_node)
builder.add_node("contradiction_node", contradiction_node)
builder.add_node("ltm_node", ltm_node)
builder.add_node("risk_node", risk_node)
builder.add_node("compiler_node", compiler_node)

builder.add_edge(START, "router_node")
builder.add_conditional_edges("router_node", route_from_router, {
    "faiss_node": "faiss_node",
    "explain_node": "explain_node",
    "compiler_node": "compiler_node"
})

builder.add_edge("faiss_node", "contradiction_node")
builder.add_conditional_edges("contradiction_node", route_after_contradiction, {
    "ltm_node": "ltm_node",
    "compiler_node": "compiler_node"
})

builder.add_edge("ltm_node", "risk_node")
builder.add_edge("risk_node", "compiler_node")
builder.add_edge("explain_node", "compiler_node")

builder.add_conditional_edges("compiler_node", completeness_check, {
    "router_node": "router_node",
    END: END
})

# Compile with Short-Term checkpoint memory
checkpointer = MemorySaver()
app = builder.compile(checkpointer=checkpointer)
print("ClauseIQ LangGraph StateGraph compiled successfully with MemorySaver checkpointing!")

In [ ]:
# Multi-intent invocation demonstrating dynamic routing, dual memory, and reflection loop
test_clause_id = "clause_17"
test_clause_text = clauses[test_clause_id]
thread_id = f"session_{test_clause_id}"

sample_state = {
    "contract_type": active_doc_type,
    "input_clause_id": test_clause_id,
    "input_clause_text": test_clause_text,
    "user_query": "Explain this clause simply and check if there is any contradiction or high risk.",
    "thread_id": thread_id,
    "intents": [],
    "visited_nodes": [],
    "plain_summary": None,
    "faiss_candidates": None,
    "contradictions": None,
    "ltm_benchmark": None,
    "risk_assessment": None,
    "final_response": None
}

config = {"configurable": {"thread_id": thread_id}}
result = app.invoke(sample_state, config=config)

print("=== LANGGRAPH EXECUTION COMPLETED ===")
print("Intents Detected:", result["intents"])
print("Execution Path Trace:", " -> ".join(result["visited_nodes"]))
print("\n=== COMPILED DEBUG JSON RESPONSE ===")
print(json.dumps(result["final_response"], indent=2, ensure_ascii=False))

In [49]:
# ClauseIQ MVP interface (run after all setup/model/tool cells above)
# A small Gradio UI that produces the same five sections on every run.

import json
import re
import gradio as gr

def _safe_tool_invoke(tool_obj, clause_text):
    """Run a LangChain tool and return either its result or a safe error."""
    try:
        result = tool_obj.invoke({"clause": clause_text})
        return {"ok": True, "result": result}
    except Exception as exc:
        print(f"{getattr(tool_obj, 'name', 'tool')} failed: {type(exc).__name__}: {exc}")
        return {"ok": False, "error": "This analysis could not be completed."}

def _as_json(value):
    if isinstance(value, dict):
        return value
    try:
        return json.loads(str(value))
    except (TypeError, json.JSONDecodeError):
        return None

def analyze_clause_mvp(clause_id):
    """Run each analysis independently and render a stable, guarded result."""
    if clause_id not in clauses:
        return "### Input error\nPlease select a clause from the loaded contract."

    clause_text = clauses[clause_id]
    risk_result = _safe_tool_invoke(classify_clause_risk, clause_text)
    summary_result = _safe_tool_invoke(summarize_clause, clause_text)
    related_result = _safe_tool_invoke(find_related_clauses, clause_text)
    contradiction_result = _safe_tool_invoke(check_clause_contradictions, clause_text)

    sections = [
        f"# ClauseIQ — {clause_id}",
        "> **Review aid only:** AI-generated analysis, not legal advice. Verify important findings against the complete contract.",
        "## Selected clause",
        f"> {clause_text}",
        "## 1. Risk classification",
    ]

    if risk_result["ok"]:
        risk_text = str(risk_result["result"]).strip()
        match = re.search(r"\b(LOW|MEDIUM|HIGH)\b", risk_text, flags=re.I)
        if match:
            sections.append(f"**Risk level: {match.group(1).upper()}**")
        else:
            sections.append("**Risk level: Unavailable / unrecognized output**")
        sections.append(risk_text)
    else:
        sections.append("Risk analysis failed. No risk category is being reported.")

    sections.append("## 2. Simplified explanation")
    if summary_result["ok"] and str(summary_result["result"]).strip():
        sections.append(str(summary_result["result"]).strip())
    else:
        sections.append("Summarization failed. No explanation is being inferred.")

    related_data = _as_json(related_result.get("result")) if related_result["ok"] else None
    sections.append("## 3. Related clauses")
    if not related_result["ok"] or related_data is None or related_data.get("error"):
        sections.append("Related-clause analysis failed. No relationship is being assumed.")
    else:
        related = related_data.get("related_clauses", [])
        if not related:
            sections.append("No related clauses were identified by the current analysis. This does not prove that none exist.")
        else:
            for item in related:
                sections.append(
                    f"- **{item.get('clause_id', 'Clause ID unavailable')}** "
                    f"({item.get('relationship_type', 'relationship unspecified')}; "
                    f"similarity: {item.get('similarity', 'n/a')}) — "
                    f"{item.get('reason', 'No reason supplied.')}\n\n"
                    f"  > {item.get('text', 'Clause text unavailable')}"
                )

    contradiction_data = _as_json(contradiction_result.get("result")) if contradiction_result["ok"] else None
    sections.append("## 4. Potential contradictions")
    if not contradiction_result["ok"] or contradiction_data is None or contradiction_data.get("error"):
        sections.append("Contradiction screening failed. No conclusion is being inferred.")
    else:
        contradictions = contradiction_data.get("contradictions", [])
        if contradictions:
            sections.append(
                f"**{len(contradictions)} potential contradiction(s) flagged** "
                f"across {contradiction_data.get('related_clauses_checked', 'an unknown number of')} related clause(s)."
            )
            for item in contradictions:
                score = item.get("contradiction_score")
                score_text = f"{float(score):.3f}" if isinstance(score, (int, float)) else "not available"
                sections.append(
                    f"- **{item.get('clause_id', 'Clause ID unavailable')}** — "
                    f"potential contradiction score: {score_text}. "
                    f"{item.get('relationship_reason', '')}\n\n"
                    f"  > {item.get('text', 'Clause text unavailable')}"
                )
            sections.append(
                "_These are model-generated flags for human review, not confirmed legal contradictions. "
                "A score is not a probability of legal error._"
            )
        else:
            sections.append(
                "No potential contradiction was flagged among the related clauses checked. "
                "This is not a guarantee that the contract is consistent."
            )

    sections.append("## 5. Limitations")
    sections.append(
        "- Results depend on text extraction, retrieval, and model performance.\n"
        "- Review the full contract and verify clause references, amounts, dates, and obligations.\n"
        "- ClauseIQ does not replace advice from a qualified legal professional."
    )
    return "\n\n".join(sections)

with gr.Blocks(title="ClauseIQ MVP") as clauseiq_demo:
    gr.Markdown(
        "# ClauseIQ\n"
        "Contract-clause review: risk, plain-language summary, related clauses, and potential contradictions.\n\n"
        "**Use only as a review aid; not legal advice.**"
    )
    clause_picker = gr.Dropdown(
        choices=[(f"{cid}: {text[:110]}{'…' if len(text) > 110 else ''}", cid)
                 for cid, text in clauses.items()],
        label="Choose a clause from the loaded contract",
        value=clause_ids[0] if clause_ids else None
    )
    analyze_button = gr.Button("Analyze clause", variant="primary")
    output_markdown = gr.Markdown()
    analyze_button.click(fn=analyze_clause_mvp, inputs=clause_picker, outputs=output_markdown)

# In a notebook, this opens the local Gradio interface. Set share=True only if
# you deliberately intend to expose a temporary public link.
clauseiq_demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


RAW LLM RESPONSE: '{\n  "related_clauses": [\n    {\n      "clause_id": "clause_15",\n      "reason": "Specifies the condition at the end of the lease period and the refund of advance, directly affecting the lease granted in the primary clause.",\n      "relationship_type": "implements"\n    },\n    {\n      "clause_id": "clause_6",\n      "reason": "States that the lessor owns and possesses the property, providing necessary context for the lessor’s ability to grant the lease.",\n      "relationship_type": "context"\n    },\n    {\n      "clause_id": "clause_12",\n      "reason": "Imposes a restriction on sub‑leasing, which is a term of the lease granted in the primary clause.",\n      "relationship_type": "implements"\n    },\n    {\n      "clause_id": "clause_8",\n      "reason": "Details the advance payment received and its refund conditions, a term of the lease granted in the primary clause.",\n      "relationship_type": "implements"\n    },\n    {\n      "clause_id": "clause_9",\n